# M2 — Clustering GPU (cuML) vs CPU (Scikit-Learn) — Entrega 3
## Sistema de Recomendación Paralelo para E-Commerce — RetailRocket Dataset

**Corre en Google Colab con GPU activada.**

Cubre los criterios de Entrega 3 de M2:
1. Clustering final con cuML (GPU) vs Scikit-Learn (CPU) sobre el dataset completo de usuarios.
2. Benchmark de speedup GPU vs CPU para 100K, 1M y 10M de registros — identificando el umbral donde GPU compensa su overhead.
3. Reducción dimensional con UMAP (GPU) para visualización 2D.
4. Interpretación de los segmentos resultantes.

⚠️ Igual que con cuDF en M1: esta es la primera corrida de cuML en este
proyecto, no se pudo probar de antemano por falta de GPU en el entorno de
desarrollo. Si algo falla, comparte el error exacto tal como con los
notebooks anteriores.


## 0. Configuración inicial

In [ ]:
import torch
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Dispositivo:", torch.cuda.get_device_name(0))


### Instalar cuDF + cuML (RAPIDS)

In [ ]:
!pip install --extra-index-url=https://pypi.nvidia.com cudf-cu12 cuml-cu12


### Traer el código del repositorio

In [ ]:
!git clone https://github.com/DavidMoraV/Proyecto-Paralela_E-commerce.git
%cd Proyecto-Paralela_E-commerce/Codigos
!pip install -q polars scikit-learn


### Subir los datos

Sube `eventos_limpios.zip` (igual que en el notebook de M3) para reconstruir
las features de usuario.


In [ ]:
from google.colab import files
import zipfile
from pathlib import Path

Path("datos").mkdir(exist_ok=True)
print("Sube eventos_limpios.zip:")
subido = files.upload()
with zipfile.ZipFile(list(subido.keys())[0]) as z:
    z.extractall("datos")


## 1. Cargar datos y construir features de usuario

In [ ]:
import sys
sys.path.append("src")

from pipeline_datos import cargar_eventos_procesados
from analisis_eda import construir_features_usuario
from pathlib import Path

eventos = cargar_eventos_procesados(Path("datos")).collect()
features_usuario = construir_features_usuario(eventos.lazy())
print(f"Usuarios: {features_usuario.height:,}")
features_usuario.head()


## 2. Clustering sobre el dataset real: CPU vs GPU

Mismo preprocesamiento (log1p + estandarización + K-Means, k=6) que en la
Entrega 2, corrido una vez en CPU y una vez en GPU, sobre los ~1.4M usuarios
reales.


In [ ]:
from analisis_eda_gpu import segmentar_usuarios_cpu_referencia, segmentar_usuarios_gpu

resultado_cpu, tiempo_cpu = segmentar_usuarios_cpu_referencia(features_usuario, n_clusters=6)
print(f"CPU: {tiempo_cpu:.2f}s")

resultado_gpu, tiempo_gpu = segmentar_usuarios_gpu(features_usuario, n_clusters=6)
print(f"GPU: {tiempo_gpu:.2f}s")
print(f"Speedup GPU sobre el dataset real ({features_usuario.height:,} usuarios): {tiempo_cpu/tiempo_gpu:.2f}x")


## 3. Benchmark de escalabilidad: 100K, 1M y 10M de registros

Usa datos sintéticos (con la misma distribución que las features reales)
porque el dataset real (~1.4M usuarios) no alcanza los 10M de filas que
pide explorar la rúbrica. El objetivo es identificar el **umbral** de
tamaño a partir del cual GPU compensa su overhead de transferencia.


In [ ]:
from analisis_eda_gpu import benchmark_kmeans_gpu_vs_cpu

resultado_benchmark = benchmark_kmeans_gpu_vs_cpu(
    tamanos=[100_000, 1_000_000, 10_000_000], n_clusters=6, n_repeticiones=3
)

Path("resultados/entrega3").mkdir(parents=True, exist_ok=True)
resultado_benchmark.to_csv("resultados/entrega3/benchmark_kmeans_gpu_vs_cpu.csv", index=False)
resultado_benchmark


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(resultado_benchmark["n_filas"], resultado_benchmark["tiempo_cpu_s"], marker="o", label="CPU", color="#5F5E5A")
axes[0].plot(resultado_benchmark["n_filas"], resultado_benchmark["tiempo_gpu_s"], marker="o", label="GPU", color="#0F6E56")
axes[0].set_xscale("log"); axes[0].set_yscale("log")
axes[0].set_xlabel("Filas"); axes[0].set_ylabel("Tiempo (s)"); axes[0].set_title("K-Means: CPU vs GPU"); axes[0].legend()

axes[1].plot(resultado_benchmark["n_filas"], resultado_benchmark["speedup_gpu"], marker="o", color="#993C1D")
axes[1].axhline(1.0, linestyle="--", color="gray")
axes[1].set_xscale("log")
axes[1].set_xlabel("Filas"); axes[1].set_ylabel("Speedup GPU"); axes[1].set_title("Speedup vs tamaño del dataset")

plt.tight_layout()
plt.savefig("resultados/entrega3/benchmark_kmeans_gpu_vs_cpu.png", dpi=150, bbox_inches="tight")
plt.show()


**Interpretación del umbral:** el punto donde `speedup_gpu` cruza 1.0x (columna
`gpu_mas_rapida`) es el volumen de datos a partir del cual GPU compensa su
overhead de transferencia CPU→GPU. Para volúmenes menores, CPU puede ser
igual o más rápido -- el mismo principio ya observado con cuDF en M1 y con
el modelo NCF en M3 (modelos/datasets pequeños no aprovechan bien la GPU).


## 4. Reducción dimensional (UMAP, GPU) para visualización 2D

In [ ]:
from analisis_eda_gpu import reducir_umap_gpu

embedding_2d = reducir_umap_gpu(resultado_gpu, n_muestra=50_000)
embedding_2d.to_csv("resultados/entrega3/umap_2d_usuarios.csv", index=False)
embedding_2d.head()


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(embedding_2d["umap_x"], embedding_2d["umap_y"], c=embedding_2d["cluster"], cmap="tab10", s=3, alpha=0.5)
legend = ax.legend(*scatter.legend_elements(), title="Cluster", loc="best")
ax.add_artist(legend)
ax.set_title("Proyección UMAP 2D de usuarios, coloreada por cluster")
ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
plt.savefig("resultados/entrega3/umap_2d_usuarios.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. Análisis de resultados — interpretación de segmentos

*(Completar con la interpretación real de la proyección UMAP: si los
clusters aparecen como regiones separadas en el plano 2D, confirma que las
6 variables de features capturan diferencias reales de comportamiento entre
segmentos -- si se solapan mucho, sugiere que algunos clusters son más
similares entre sí de lo que el número k=6 asume.)*

**Implicaciones para el sistema de recomendación:** ver la sección de M3
(Entrega 2/3) donde ya se documentó que el Cluster 2 (usuarios muy activos)
tiene paradójicamente el peor Hit Rate del modelo ALS/NCF, por explorar un
catálogo más amplio y diverso. La proyección UMAP de esta sección permite
verificar visualmente si ese cluster ocupa una región densa y distintiva del
espacio de features, o si su comportamiento es más disperso de lo que
sugiere una sola etiqueta de cluster.


## 6. Descargar resultados

In [ ]:
from google.colab import files

files.download("resultados/entrega3/benchmark_kmeans_gpu_vs_cpu.csv")
files.download("resultados/entrega3/umap_2d_usuarios.csv")
files.download("resultados/entrega3/umap_2d_usuarios.png")
